In [0]:
# Check what's available
dbutils.fs.ls("/")

[FileInfo(path='dbfs:/Volumes/', name='Volumes/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/Workspace/', name='Workspace/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/databricks-datasets/', name='databricks-datasets/', size=0, modificationTime=0)]

In [0]:
# Read the table created
df = spark.table("default.car_sales_data")

# Or if seen in a different schema:
df = spark.sql("SELECT * FROM car_sales_data")

display(df.limit(10))

Manufacturer,Model,Engine size,Fuel type,Year of manufacture,Mileage,Price
Ford,Fiesta,1.0,Petrol,2002,127300,3074
Porsche,718 Cayman,4.0,Petrol,2016,57850,49704
Ford,Mondeo,1.6,Diesel,2014,39190,24072
Toyota,RAV4,1.8,Hybrid,1988,210814,1705
VW,Polo,1.0,Petrol,2006,127869,4101
Ford,Focus,1.4,Petrol,2018,33603,29204
Ford,Mondeo,1.8,Diesel,2010,86686,14350
Toyota,Prius,1.4,Hybrid,2015,30663,30297
VW,Polo,1.2,Petrol,2012,73470,9977
Ford,Focus,2.0,Diesel,1992,262514,1049


In [0]:
# =====================================================
# PHASE 1: Data Pipeline & ETL
# =====================================================

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Load Data
df = spark.table("car_sales_data")

display(df.limit(10))
print(f"Total records: {df.count()}")
print(f"Columns: {df.columns}")

# Data Quality Check
print("\n=== Missing Values ===")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# Clean column names
df_renamed = df \
    .withColumnRenamed("Year of manufacture", "year") \
    .withColumnRenamed("Fuel type", "fuel_type") \
    .withColumnRenamed("Engine size", "engine_size") \
    .withColumnRenamed("Manufacturer", "brand") \
    .withColumnRenamed("Model", "model") \
    .withColumnRenamed("Price", "price") \
    .withColumnRenamed("Mileage", "mileage")

# Clean and transform
df_cleaned = df_renamed \
    .dropDuplicates() \
    .na.drop(subset=["price", "year", "mileage"]) \
    .filter(col("price") > 0) \
    .filter(col("year") >= 2000)

# Create features
df_transformed = df_cleaned \
    .withColumn("car_age", lit(2025) - col("year")) \
    .withColumn("price_category", 
                when(col("price") < 10000, "Budget")
                .when(col("price") < 30000, "Mid-Range")
                .otherwise("Luxury")) \
    .withColumn("mileage_category",
                when(col("mileage") < 30000, "Low")
                .when(col("mileage") < 80000, "Medium")
                .otherwise("High"))

# Create database and save
spark.sql("CREATE DATABASE IF NOT EXISTS hackathon_db")

df_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("hackathon_db.car_sales_clean")

print("\n✅ Data pipeline complete!")

# Summary
df_transformed.select("price", "mileage", "car_age").summary().show()
display(df_transformed.select("brand", "model", "year", "price", "mileage", "car_age", "price_category").limit(10))

spark.sql("SELECT COUNT(*) as total_records FROM hackathon_db.car_sales_clean").show()
df_transformed.groupBy("brand").count().orderBy(desc("count")).show(10)

Manufacturer,Model,Engine size,Fuel type,Year of manufacture,Mileage,Price
Ford,Fiesta,1.0,Petrol,2002,127300,3074
Porsche,718 Cayman,4.0,Petrol,2016,57850,49704
Ford,Mondeo,1.6,Diesel,2014,39190,24072
Toyota,RAV4,1.8,Hybrid,1988,210814,1705
VW,Polo,1.0,Petrol,2006,127869,4101
Ford,Focus,1.4,Petrol,2018,33603,29204
Ford,Mondeo,1.8,Diesel,2010,86686,14350
Toyota,Prius,1.4,Hybrid,2015,30663,30297
VW,Polo,1.2,Petrol,2012,73470,9977
Ford,Focus,2.0,Diesel,1992,262514,1049


Total records: 50000
Columns: ['Manufacturer', 'Model', 'Engine size', 'Fuel type', 'Year of manufacture', 'Mileage', 'Price']

=== Missing Values ===
+------------+-----+-----------+---------+-------------------+-------+-----+
|Manufacturer|Model|Engine size|Fuel type|Year of manufacture|Mileage|Price|
+------------+-----+-----------+---------+-------------------+-------+-----+
|           0|    0|          0|        0|                  0|      0|    0|
+------------+-----+-----------+---------+-------------------+-------+-----+


✅ Data pipeline complete!
+-------+------------------+-----------------+------------------+
|summary|             price|          mileage|           car_age|
+-------+------------------+-----------------+------------------+
|  count|             32385|            32385|             32385|
|   mean|19760.653913849004|77164.39048942411|14.918388142658639|
| stddev|17683.385475904524| 46373.6577537205| 6.131649461541892|
|    min|               697|            

brand,model,year,price,mileage,car_age,price_category
Ford,Fiesta,2002,3074,127300,23,Budget
Porsche,718 Cayman,2016,49704,57850,9,Luxury
Ford,Mondeo,2014,24072,39190,11,Mid-Range
VW,Polo,2006,4101,127869,19,Budget
Ford,Focus,2018,29204,33603,7,Mid-Range
Ford,Mondeo,2010,14350,86686,15,Mid-Range
Toyota,Prius,2015,30297,30663,10,Luxury
VW,Polo,2012,9977,73470,13,Budget
VW,Golf,2014,17173,83047,11,Mid-Range
VW,Golf,2007,7792,92697,18,Budget


+-------------+
|total_records|
+-------------+
|        32385|
+-------------+

+-------+-----+
|  brand|count|
+-------+-----+
|   Ford| 9726|
|     VW| 9621|
| Toyota| 8142|
|    BMW| 3169|
|Porsche| 1727|
+-------+-----+

